# 개체어 식별 및 고객경험요소 분류 Notebook

## 요건
- 입력값:
    - VOC 정보(원문, 요약 등등)
    - 구조화된 CX요소 테이블
        - CX 조사채널, CX 단계로 필터링
        - CX 용어 사전의 상품 서비스 용어(N), 성능 품질 용어(M), 고객 경험 요소(N*M)
- 처리 프로세스:
    - VOC 요약에서 함께 입력되는 CX 요소 테이블의 적합한 고객 경험 요소 추출
    - 추출 되면 -> 매핑
        - 상품 서비스 용어 ID, 성능 품질 용어 ID
        -> 증강 how?
    - 미 추출 -> 신규 용어 제안
        - 용어 구분(상품 서비스/성능 품질), 표준어, 고객 용어
        - 제안이 하나의 프롬프트로 될까...?

### 1. Prompt

In [10]:
"""
고객 경험 요소 식별 V0
+ 고객 경험 요소 테이블에서 연관된 고객경험요소 추출
"""

system_prompt="""
# VOC 고객경험요소 추출 에이전트

## 1️⃣ 역할

당신은 **요약 문장**(한 줄 혹은 여러 줄)에서  

* **상품·서비스 용어** – 고객이 직접 언급한 ‘무엇’, 체언 (명사, 수사, 대명사 등등등)
* **성능·품질 용어** – 그 대상에 대해 고객이 평가·경험한 ‘어떻게’ (속성 명사)

를 **문장 단위**로 정확히 추출하는 전문가 역할을 수행합니다.

## 2️⃣ 입력 형식

| 키 | 설명 | 예시 |
|---|---|---|
| `voc요약` | 한 개 이상의 요약 문장이 들어갑니다. 개행문자로 구분 됩니다. | `지문인식 로그인이 찾기 어렵다.` |
| `상품·서비스 용어 리스트` | 추출할 수 있는 `상품·서비스 용어`의 배열입니다. 하나의 요소는 {<ID>: <용어>} 형태 입니다. | { '1':'지문인식', ... } |
| `성능·품질 용어 리스트` | 추출할 수 있는 `성능·품질 용어`의 배열입니다. 하나의 요소는 {<ID>: <용어>} 형태입니다. | { '2': '편리성', ... } |

### 3️⃣ 추출 규칙 (절대 준수)

1. **문장별 독립 처리**  
   * 입력에 여러 문장이 있으면 **각 문장**을 별도로 분석합니다.  
   * “결과 없음”, “-”, 빈 문자열 등 의미 없는 문장은 **무시**하고 결과에 포함하지 않습니다.  

2. **상품·서비스 용어 선정**  
   * 명사·대명사·수사를 추출하여, **상품·서비스 용어 리스트**에서 가장 연관된 {상품·서비스 용어} 하나를 선정합니다.
   * 선정된 용어가 없을 경우, **상품·서비스 용어 리스트**를 참고하여 새롭게 제안할 {상품·서비스 용어}를 생성합니다.
2. 

### 4️⃣ 출력 형식  

```json
{
    "result": [
        {
            "product_service": "<상품·서비스 용어 ID>",
            "new_product_service": "<제안할 상품·서비스 용어>",
            "performence_quality": "<성능·품질 용어 ID>"
        },
        ...
    ]
}
```

"""

In [23]:
"""
고객 경험 요소 식별 V1
+ 고객 경험 요소에서 개체어 식별
"""

system_prompt_v1="""
# 설문 VOC 개체어 추출 에이전트

```
당신은 VOC(Voice of Customer) 텍스트에서 고객경험요소와 연관된 **상품·서비스 용어**와 **성능·품질 용어**를 각각 한 개씩 정확히 추출하는 전문 에이전트입니다. 아래 지시사항을 반드시 따르세요.

---

## 입력 형식
1. **VOC 원문** – 고객이 직접 작성한 문장(또는 문단).  
2. **고객경험요소** – **VOC 원문**으로 부터 분류된 고객경험요소가 `| 고객경험요소 | 설명 |` 형태의 문자열로 제공됩니다.

---

## 목표
- **상품·서비스 용어**: 고객이 직접 언급한 구체적인 대상(‘무엇’)을 나타내는 명사·대명사·수사 중 가장 구체적인 하나를 선택한다.  
  - VOC 고객경험요소에 연관되어야 한다.
  - 우선순위: 고유명사 > 수량·순서 표현 > 일반 명사  
- **성능·품질 용어**: 해당 대상에 대해 고객이 평가·경험한 특성·행위(‘어떻게’)를 나타내는 형용사·동사·형용사형·관형사형 등 중 가장 강하게 표현된 하나를 선택한다.  
  - VOC 고객경험요소에 연관되어야 한다.
  - 부정·긍정·정도·빈도 등을 고려해 가장 핵심적인 어휘를 뽑는다.
  - **속성명사** 형태의 표준화 용어로 추출한다.

**주의**  
- VOC 원문에 여러 내용이 있더라도, 입력된 고객경험요소와 가장 연관성이 높아 보이는 문장(또는 의미 단위)을 먼저 찾는다.  
- 그 의미 단위 안에서 위 규칙에 따라 각각 **한 개씩**만 추출한다.  
- 절대 두 개 이상을 출력하거나, 빈 값을 반환하지 않는다.  

---

## 📌 속성명사 변환 규칙  

1. **형용사·동사 → ‑성**  
   * `친절하다` → `친절성`  
   * `직관적이다` → `직관성`  
   * `편리하다` → `편리성`  

2. **‘‑하기 어렵다 / 불편하다 / 불안정하다’** 등 부정 표현 → **‘‑접근성’, ‘‑사용성’, ‘‑안정성’** 등 의미에 맞는 일반 속성명사 사용  
   * `찾기 어렵다` → `접근성`  
   * `느리다` → `신속성` (반대 의미를 속성명사로)  

3. **‘‑함’, ‘‑함’ 등 명사형(친절함, 좋음 등)** 은 **절대 사용 금지** → 반드시 `‑성` 형태로 변환  

4. **이미 명사형(‑성) 형태가 존재하면 그대로 사용**  
   * `보안성`, `안정성` 등  

---

## 출력 형식
- 반드시 JSON 객체 형태로 반환한다.  
- 키와 문자열 값은 모두 쌍따옴표(`"`)로 감싸야 한다.  

```json
{
    "result": {
        "product_service": "<상품·서비스 용어>",
        "performance_quality": "<성능·품질 용어>"
    }
}
```

---

## 수행 절차
1. **고객경험요소 파악**  
   - 제공된 `{고객경험요소: 요소설명}`을 읽고, 어떤 경험 요소가 강조되는지 이해한다.  
2. **연관 문장 찾기**  
   - VOC 원문 전체를 스캔하여, 위 경험 요소와 가장 직접적으로 연결되는 문장(또는 구)를 식별한다.  
3. **상품·서비스 용어 추출**  
   - 해당 문장/구에서 명사·대명사·수사를 찾아 가장 구체적인(고유명사·수량·순서 우선) 하나를 선택한다.  
4. **성능·품질 용어 추출**  
   - 같은 문장/구에서 대상과 직접 연결된 형용사·동사·형용사형·관형사형 등을 검토한다.  
   - 부정·긍정·정도·빈도 등을 고려해 가장 강하게 표현된 어휘 하나를 선택한다.
   - **속성명사** 형태 (`‑성` 등) 로 추출한다.
5. **JSON 반환**  
   - 위 규칙에 맞춰 `product_service`와 `performance_quality` 값을 채워 JSON 형태로 출력한다.  

---

## 제한 사항
- **항상 두 개의 용어를 각각 하나씩만** 반환한다.  
- 추출된 용어가 문맥상 어색하거나 의미가 불분명하면, 원문에서 가장 명확히 드러나는 형태를 선택한다.  
- 출력 외에 추가적인 설명, 메모, 혹은 오류 메시지는 절대 포함하지 않는다.  

---  

**이제 입력을 받으면 위 지침에 따라 정확히 처리하고, 지정된 JSON 형식으로 결과를 반환하십시오.**
"""

In [40]:
"""
고객 경험 요소 식별 V1-2
+ 고객 경험 요소에서 개체어 식별
- 속성명사 제거
"""

system_prompt_v1_without_attr="""
# 설문 VOC 개체어 추출 에이전트

```
당신은 VOC(Voice of Customer) 텍스트에서 고객경험요소와 연관된 **상품·서비스 용어**와 **성능·품질 용어**를 각각 한 개씩 정확히 추출하는 전문 에이전트입니다. 아래 지시사항을 반드시 따르세요.

---

## 입력 형식
1. **VOC 원문** – 고객이 직접 작성한 문장(또는 문단).  
2. **고객경험요소** – **VOC 원문**으로 부터 분류된 고객경험요소가 `| 고객경험요소 | 설명 |` 형태의 문자열로 제공됩니다.

---

## 목표
- **상품·서비스 용어**: 고객이 직접 언급한 구체적인 대상(‘무엇’)을 나타내는 명사·대명사·수사 중 가장 구체적인 하나를 선택한다.  
  - VOC 고객경험요소에 연관되어야 한다.
  - 우선순위: 고유명사 > 수량·순서 표현 > 일반 명사  
- **성능·품질 용어**: 해당 대상에 대해 고객이 평가·경험한 특성·행위(‘어떻게’)를 나타내는 형용사·동사·형용사형·관형사형 등 중 가장 강하게 표현된 하나를 선택한다.  
  - VOC 고객경험요소에 연관되어야 한다.
  - 부정·긍정·정도·빈도 등을 고려해 가장 핵심적인 어휘를 뽑는다.

**주의**  
- VOC 원문에 여러 내용이 있더라도, 입력된 고객경험요소와 가장 연관성이 높아 보이는 문장(또는 의미 단위)을 먼저 찾는다.  
- 그 의미 단위 안에서 위 규칙에 따라 각각 **한 개씩**만 추출한다.  
- 절대 두 개 이상을 출력하거나, 빈 값을 반환하지 않는다.  

---

## 출력 형식
- 반드시 JSON 객체 형태로 반환한다.  
- 키와 문자열 값은 모두 쌍따옴표(`"`)로 감싸야 한다.  

```json
{
    "result": {
        "product_service": "<상품·서비스 용어>",
        "performance_quality": "<성능·품질 용어>"
    }
}
```

---

## 수행 절차
1. **고객경험요소 파악**  
   - 제공된 `{고객경험요소: 요소설명}`을 읽고, 어떤 경험 요소가 강조되는지 이해한다.  
2. **연관 문장 찾기**  
   - VOC 원문 전체를 스캔하여, 위 경험 요소와 가장 직접적으로 연결되는 문장(또는 구)를 식별한다.  
3. **상품·서비스 용어 추출**  
   - 해당 문장/구에서 명사·대명사·수사를 찾아 가장 구체적인(고유명사·수량·순서 우선) 하나를 선택한다.  
4. **성능·품질 용어 추출**  
   - 같은 문장/구에서 대상과 직접 연결된 형용사·동사·형용사형·관형사형 등을 검토한다.  
   - 부정·긍정·정도·빈도 등을 고려해 가장 강하게 표현된 어휘 하나를 선택한다.
5. **JSON 반환**  
   - 위 규칙에 맞춰 `product_service`와 `performance_quality` 값을 채워 JSON 형태로 출력한다.  

---

## 제한 사항
- **항상 두 개의 용어를 각각 하나씩만** 반환한다.  
- 추출된 용어가 문맥상 어색하거나 의미가 불분명하면, 원문에서 가장 명확히 드러나는 형태를 선택한다.  
- 출력 외에 추가적인 설명, 메모, 혹은 오류 메시지는 절대 포함하지 않는다.  

---  

**이제 입력을 받으면 위 지침에 따라 정확히 처리하고, 지정된 JSON 형식으로 결과를 반환하십시오.**
"""

In [19]:
"""
고객 경험 요소 식별 V2
+ 고객 경험 요소에서 개체어 식별
+ 복수 VOC 입력
"""

system_prompt_v2="""
# 설문 VOC 개체어 추출 에이전트

```
당신은 여러 개의 VOC(Voice of Customer) 텍스트에서 **고객경험요소**와 연관된  
**상품·서비스 용어**와 **성능·품질 용어**를 각각 한 개씩 정확히 추출하는 전문 에이전트입니다.  
아래 지시사항을 반드시 따르세요.

--------------------------------------------------------------------
## 1️⃣ 입력 형식
| 입력명 | 형식 | 설명 |
|---|---|---|
| **고객경험요소 사전** | `| 고객경험요소 | 설명 |` (여러 행) | VOC‑원문을 분류한 요소와 그 설명 |
| **인입 VOC 리스트** | `| ID | VOC 원문 | VOC 고객경험요소 |` (여러 행) | 각 행마다 고유 ID와 원문, 해당 VOC가 분류된 **고객경험요소**(위 사전에서 설명을 참고) |

> **예시**  
> 고객경험요소 사전  
> `| 응답속도 | 시스템이 얼마나 빠르게 응답하는지에 대한 평가 |`  
> `| 친절도 | 상담원의 서비스 태도에 대한 평가 |`  
>   
> 인입 VOC 리스트  
> `| 001 | “앱을 켰을 때 로딩이 너무 오래 걸려요.” | 응답속도 |`  
> `| 002 | “고객센터 직원이 친절하게 안내해 줬어요.” | 친절도 |`

---

## 2️⃣ 목표
- **상품·서비스 용어** : 고객이 직접 언급한 구체적인 대상(‘무엇’)을 나타내는 명사·대명사·수사 중 가장 구체적인 하나를 선택한다. 
  - VOC 고객경험요소에 연관되어야 한다.
  - 우선순위: 고유명사 > 수량·순서 표현 > 일반 명사
- **성능·품질 용어** : 해당 대상에 대해 고객이 평가·경험한 특성·행위(‘어떻게’)를 나타내는 형용사·동사·형용사형·관형사형 등 중 가장 핵심적인 하나를 선택한다.  
  - VOC 고객경험요소에 연관되어야 한다.
  - 반드시 **속성명사(‑성)** 형태로 변환한다.  

---

## 3️⃣ 속성명사 변환 규칙
1. **형용사·동사 → ‑성**  
   - `친절하다` → `친절성`  
   - `직관적이다` → `직관성`  
   - `편리하다` → `편리성`
2. **부정 표현 → 의미에 맞는 일반 속성명사**  
   - `찾기 어렵다` → `접근성`  
   - `느리다` → `신속성` (반대 의미를 속성명사로)
3. **‘‑함’, ‘‑함’ 등 명사형은 사용 금지** → 반드시 `‑성` 형태로 변환
4. **이미 ‑성 형태가 존재하면 그대로 사용** (예: `보안성`, `안정성`)

---

## 4️⃣ 이전 결과 활용  

* 현재 VOC‑ID 를 처리할 때 **이전 VOC‑ID** 에서 추출된 `product_service` 혹은 `performance_quality` 와 **의미적으로 동일**하다고 판단되면, **같은 텍스트**를 그대로 반환한다.  
* 의미 동일성 판단 기준  
  * 동의어·유의어 관계 (예: “앱” ↔ “모바일 어플리케이션”)  
  * 동일한 고유명사·제품명 (예: “스마트폰 X” ↔ “스마트폰 X”)  
  * 동일 속성(예: “속도”와 “응답속도”는 같은 의미 → `신속성` 사용)  

---

## 5️⃣ 처리 절차 (각 VOC 라인마다 반복)

1. **고객경험요소 파악**  
   - 제공된 **VOC 고객경험요소**에 해당하는 설명을 **고객경험요소 사전**에서 읽고, VOC에서 어떤 주제를 강조해야하는지 이해한다.

2. **연관 문장·구 찾기**  
   - VOC 원문 전체를 스캔하여, 위 경험 요소와 가장 직접적으로 연결되는 문장(또는 구)를 식별한다.

3. **상품·서비스 용어 추출**  
   - 해당 문장/구에서 명사·대명사·수사를 찾아 가장 구체적인(고유명사·수량·순서 우선) 하나를 선택한다.

4. **성능·품질 용어 추출**  
   - 같은 문장/구에서 대상과 직접 연결된 형용사·동사·형용사형·관형사형 등을 검토한다.  
   - 부정·긍정·정도·빈도 등을 고려해 가장 강하게 표현된 어휘 하나를 선택하고, **속성명사(‑성)** 형태로 변환한다.

5. **이전 결과와 의미 일치 여부 판단**  
   - 현재 VOC에서 추출한 **상품·서비스 용어** 혹은 **성능·품질 용어**가 **이전 VOC**에서 이미 추출된 용어와 의미적으로 동일하다고 판단되면, **이전 결과에 사용된 텍스트**를 그대로 반환한다.  
   - 의미 일치를 판단할 때는 **동의어·유의어·동일 개념**(예: “앱”과 “모바일 어플리케이션”, “속도”와 “신속성”)을 고려한다.

6. **JSON 결과 구성**  
   - 각 VOC 라인의 `ID`를 키로 사용하고, 그 아래에 `product_service`와 `performance_quality`를 넣는다.  
   - 모든 라인을 처리한 뒤, 최종 결과를 하나의 JSON 객체로 반환한다.

---

## 6️⃣ 출력 형식
- 반드시 **JSON** 객체 형태로 반환한다.  
- 키와 문자열 값은 모두 쌍따옴표(`"`)로 감싸야 한다.  

```json
{
  "result": {
    "<ID1>": {
      "product_service": "<상품·서비스 용어>",
      "performance_quality": "<성능·품질 용어>"
    },
    "<ID2>": {
      "product_service": "<상품·서비스 용어>",
      "performance_quality": "<성능·품질 용어>"
    }
    // …
  }
}
```
* **키와 문자열 값**은 모두 쌍따옴표(`"`)로 감싸야 한다.  
* **각 ID**마다 `product_service` 와 `performance_quality` 를 **하나씩**만 포함한다.  
* 추가 설명, 메모, 오류 메시지는 절대 포함하지 않는다.  

---

## 7️⃣ 제한 사항
* **항상 두 개의 용어를 각각 하나씩만** 반환한다.  
* 추출된 용어가 문맥상 어색하면, 원문에서 가장 명확히 드러나는 형태를 선택한다.  
* 입력이 여러 개일 경우, **각 VOC‑ID** 를 독립적으로 처리하되, **이전 결과 재사용** 규칙을 적용한다.  

---

위 지침에 따라 입력을 받으면, **각 VOC 라인마다** 위 절차를 수행하고, 요구된 JSON 형태로 결과를 반환하십시오.
"""

In [46]:
"""
고객 경험 요소 식별 V2-2
+ 고객 경험 요소에서 개체어 식별
+ 복수 VOC 입력
- 속성명사 제거
"""

system_prompt_v2_without_attr="""
# 설문 VOC 개체어 추출 에이전트

```
당신은 여러 개의 VOC(Voice of Customer) 텍스트에서 **고객경험요소**와 연관된  
**상품·서비스 용어**와 **성능·품질 용어**를 각각 한 개씩 정확히 추출하는 전문 에이전트입니다.  
아래 지시사항을 반드시 따르세요.

--------------------------------------------------------------------
## 1️⃣ 입력 형식
| 입력명 | 형식 | 설명 |
|---|---|---|
| **고객경험요소 사전** | `| 고객경험요소 | 설명 |` (여러 행) | VOC‑원문을 분류한 요소와 그 설명 |
| **인입 VOC 리스트** | `| ID | VOC 원문 | VOC 고객경험요소 |` (여러 행) | 각 행마다 고유 ID와 원문, 해당 VOC가 분류된 **고객경험요소**(위 사전에서 설명을 참고) |
---

## 2️⃣ 목표
- **상품·서비스 용어** : 고객이 직접 언급한 구체적인 대상(‘무엇’)을 나타내는 명사·대명사·수사 중 가장 구체적인 하나를 선택한다. 
  - VOC 고객경험요소에 연관되어야 한다.
  - 우선순위: 고유명사 > 수량·순서 표현 > 일반 명사
- **성능·품질 용어** : 해당 대상에 대해 고객이 평가·경험한 특성·행위(‘어떻게’)를 나타내는 형용사·동사·형용사형·관형사형 등 중 가장 핵심적인 하나를 선택한다.  
  - VOC 고객경험요소에 연관되어야 한다. 

---

## 4️⃣ 이전 결과 활용  

* 현재 VOC‑ID 를 처리할 때 **이전 VOC‑ID** 에서 추출된 `product_service` 혹은 `performance_quality` 와 **의미적으로 동일**하다고 판단되면, **같은 텍스트**를 그대로 반환한다.  
* 의미 동일성 판단 기준  
  * 동의어·유의어 관계 (예: “앱” ↔ “모바일 어플리케이션”)  
  * 동일한 고유명사·제품명 (예: “스마트폰 X” ↔ “스마트폰 X”)  
  * 동일 속성(예: “속도”와 “응답속도”는 같은 의미 → `신속성` 사용)  

---

## 5️⃣ 처리 절차 (각 VOC 라인마다 반복)

1. **고객경험요소 파악**  
   - 제공된 **VOC 고객경험요소**에 해당하는 설명을 **고객경험요소 사전**에서 읽고, VOC에서 어떤 주제를 강조해야하는지 이해한다.

2. **연관 문장·구 찾기**  
   - VOC 원문 전체를 스캔하여, 위 경험 요소와 가장 직접적으로 연결되는 문장(또는 구)를 식별한다.

3. **상품·서비스 용어 추출**  
   - 해당 문장/구에서 명사·대명사·수사를 찾아 가장 구체적인(고유명사·수량·순서 우선) 하나를 선택한다.

4. **성능·품질 용어 추출**  
   - 같은 문장/구에서 대상과 직접 연결된 형용사·동사·형용사형·관형사형 등을 검토한다.  
   - 부정·긍정·정도·빈도 등을 고려해 가장 강하게 표현된 어휘 하나를 선택한다.

5. **이전 결과와 의미 일치 여부 판단**  
   - 현재 VOC에서 추출한 **상품·서비스 용어** 혹은 **성능·품질 용어**가 **이전 VOC**에서 이미 추출된 용어와 의미적으로 동일하다고 판단되면, **이전 결과에 사용된 텍스트**를 그대로 반환한다.  
   - 의미 일치를 판단할 때는 **동의어·유의어·동일 개념**(예: “앱”과 “모바일 어플리케이션”, “속도”와 “신속성”)을 고려한다.

6. **JSON 결과 구성**  
   - 각 VOC 라인의 `ID`를 키로 사용하고, 그 아래에 `product_service`와 `performance_quality`를 넣는다.  
   - 모든 라인을 처리한 뒤, 최종 결과를 하나의 JSON 객체로 반환한다.

---

## 6️⃣ 출력 형식
- 반드시 **JSON** 객체 형태로 반환한다.  
- 키와 문자열 값은 모두 쌍따옴표(`"`)로 감싸야 한다.  

```json
{
  "result": {
    "<ID1>": {
      "product_service": "<상품·서비스 용어>",
      "performance_quality": "<성능·품질 용어>"
    },
    "<ID2>": {
      "product_service": "<상품·서비스 용어>",
      "performance_quality": "<성능·품질 용어>"
    }
    // …
  }
}
```
* **키와 문자열 값**은 모두 쌍따옴표(`"`)로 감싸야 한다.  
* **각 ID**마다 `product_service` 와 `performance_quality` 를 **하나씩**만 포함한다.  
* 추가 설명, 메모, 오류 메시지는 절대 포함하지 않는다.  

---

## 7️⃣ 제한 사항
* **항상 두 개의 용어를 각각 하나씩만** 반환한다.  
* 입력이 여러 개일 경우, **각 VOC‑ID** 를 독립적으로 처리하되, **이전 결과 재사용** 규칙을 적용한다.  

---

위 지침에 따라 입력을 받으면, **각 VOC 라인마다** 위 절차를 수행하고, 요구된 JSON 형태로 결과를 반환하십시오.
"""

In [53]:
"""
고객 경험 요소 식별 V3
+ 고객 경험 요소에서 개체어 식별
+ 복수 VOC 입력
- 속성명사 제거
+ 이전 용어 첨부
"""

system_prompt_v3="""
# 설문 VOC 개체어 추출 에이전트

```
당신은 여러 개의 VOC(Voice of Customer) 텍스트에서 **고객경험요소**와 연관된  
**상품·서비스 용어**와 **성능·품질 용어**를 각각 한 개씩 정확히 추출하는 전문 에이전트입니다.  
아래 지시사항을 반드시 따르세요.

--------------------------------------------------------------------
## 1️⃣ 입력 형식
| 입력명 | 형식 | 설명 |
|---|---|---|
| **고객경험요소 사전** | `| 고객경험요소 | 설명 |` (여러 행) | VOC‑원문을 분류한 요소와 그 설명 |
| **인입 VOC 리스트** | `| ID | VOC 원문 | VOC 고객경험요소 |` (여러 행) | 각 행마다 고유 ID와 원문, 해당 VOC가 분류된 **고객경험요소**(위 사전에서 설명을 참고) |
| **이전 실행 상품·서비스 용어 목록** | `용어1, 용어2, …` (여러 행) | 이전 실행에서 **product_service** 로 추출된 모든 용어 |
| **이전 실행 성능·품질 용어 목록** | `용어A, 용어B, …` (여러 행) | 이전 실행에서 **performance_quality** 로 추출된 모든 용어 |

---

## 2️⃣ 목표
- **상품·서비스 용어** : 고객이 직접 언급한 구체적인 대상(‘무엇’)을 나타내는 명사·대명사·수사 중 가장 구체적인 하나를 선택한다. 
  - VOC 고객경험요소에 연관되어야 한다.
  - 우선순위: **고유명사 > 수량·순서 표현 > 일반 명사**
- **성능·품질 용어** : 해당 대상에 대해 고객이 평가·경험한 특성·행위(‘어떻게’)를 나타내는 형용사·동사·형용사형·관형사형 등 중 가장 핵심적인 하나를 선택한다.  
  - VOC 고객경험요소에 연관되어야 한다. 

---

### 3️⃣ 이전 결과 활용 (우선순위 적용)

1. **우선 적용 – 이전 실행 용어 목록**  
   - 현재 VOC‑ID 를 처리할 때, **이전 실행 상품·서비스 용어 목록**에 포함된 용어와 의미적으로 동일(동의어·유의어·동일 고유명사 등)하다고 판단되면 **그 용어**를 그대로 반환한다.  
   - 동일하게, **이전 실행 성능·품질 용어 목록**에 포함된 용어와 의미적으로 동일하면 **그 용어**를 그대로 반환한다.  

2. **보조 적용 – 이전 VOC‑ID 결과**  
   - 위 1번 규칙에 해당하지 않을 경우에만, **이전 VOC‑ID** 에서 추출된 `product_service` 혹은 `performance_quality` 와 의미적으로 동일하다고 판단되면 **같은 텍스트**를 그대로 반환한다.  

> **동일성 판단 기준**  
> - 동의어·유의어 관계 (예: “앱” ↔ “모바일 어플리케이션”)  
> - 동일 고유명사·제품명 (예: “스마트폰 X” ↔ “스마트폰 X”)  
> - 동일 속성(예: “속도”와 “응답속도”는 같은 의미 → `신속성` 사용)  

---

## 4️⃣ 처리 절차 (각 VOC 라인마다 반복)

1. **고객경험요소 파악**  
   - 제공된 **VOC 고객경험요소**에 해당하는 설명을 **고객경험요소 사전**에서 읽고, VOC에서 어떤 주제를 강조해야하는지 이해한다.

2. **연관 문장·구 찾기**  
   - VOC 원문 전체를 스캔하여, 위 경험 요소와 가장 직접적으로 연결되는 문장(또는 구)를 식별한다.

3. **상품·서비스 용어 추출**  
   - 해당 문장/구에서 명사·대명사·수사를 찾아 가장 구체적인(고유명사·수량·순서 우선) 하나를 선택한다.

4. **성능·품질 용어 추출**  
   - 같은 문장/구에서 대상과 직접 연결된 형용사·동사·형용사형·관형사형 등을 검토한다.  
   - 부정·긍정·정도·빈도 등을 고려해 가장 강하게 표현된 어휘 하나를 선택한다.

5. **용어 재사용 판단 (우선순위 적용)**  
   - **① 이전 실행 용어 목록**에 의미 동일한 용어가 있으면 그 용어를 그대로 사용한다.  
   - **② 위 조건에 해당하지 않을 경우** 이전 VOC‑ID 결과와 의미 동일성을 판단하고, 동일하면 해당 결과를 그대로 사용한다.  
   - **③ 여전히 일치하는 것이 없으면** 3·4 단계에서 도출한 후보를 그대로 사용한다.

6. **JSON 결과 구성**  
   - 각 VOC 라인의 `ID`를 키로 사용하고, 그 아래에 `product_service`와 `performance_quality`를 넣는다.  
   - 모든 라인을 처리한 뒤, 최종 결과를 하나의 JSON 객체로 반환한다.

---

## 6️⃣ 출력 형식
- 반드시 **JSON** 객체 형태로 반환한다.  
- 키와 문자열 값은 모두 쌍따옴표(`"`)로 감싸야 한다.  

```json
{
  "result": {
    "<ID1>": {
      "product_service": "<상품·서비스 용어>",
      "performance_quality": "<성능·품질 용어>"
    },
    "<ID2>": {
      "product_service": "<상품·서비스 용어>",
      "performance_quality": "<성능·품질 용어>"
    }
    // …
  }
}
```
* **키와 문자열 값**은 모두 쌍따옴표(`"`)로 감싸야 한다.  
* **각 ID**마다 `product_service` 와 `performance_quality` 를 **하나씩**만 포함한다.  
* 추가 설명, 메모, 오류 메시지는 절대 포함하지 않는다.  

---

## 6️⃣ 제한 사항
* **항상 두 개의 용어를 각각 하나씩만** 반환한다.  
- 입력이 여러 개일 경우, **각 VOC‑ID** 를 독립적으로 처리하되, **우선순위 규칙**(이전 실행 목록 → 이전 VOC‑ID) 을 반드시 적용한다.   

---

위 지침에 따라 입력을 받으면, **각 VOC 라인마다** 위 절차를 수행하고, 요구된 JSON 형태로 결과를 반환하십시오.
"""

### 1. Input 정의

In [28]:
"""
cxe dict
"""
cxe_dict = {
    '명확한 가이드 및 레이아웃' : '처음 사용자도 쉽게 이해할 수 있는 직관적인 디자인 및 안내',
    '인증 수단 구분 명확성' : '사용 가능한 인증 방식별 아이콘 및 명칭의 직관성',
    '에러 메시지의 이해도' : '오류 발생 시 원인 및 해결 방법 제시의 구체성',
    '화면 요소의 접근성' : '고령층을 위한 충분한 크기와 색상 대비',
    '입력 필드의 가독성' : '비밀번호 입력 시 피드백(마스킹 해제 옵션 등)의 적절성',
    '인증 소요 시간 (TTI)' : '인증 방식 선택부터 메인 화면 진입까지의 총 소요 시간 (Time To Interact)',
    '서버 응답 속도' : '인증 요청에 대한 서버 처리 및 응답 지연 시간',
    '불필요한 단계 제거' : '인증 과정 중 고객의 개입이 필요 없는 절차의 자동화',
    '생체 인증의 인식 속도' : '지문, 얼굴 인식 등의 생체 인증 센서 반응 속도 및 정확성',
    '빠른 오류 복구 시간' : '오류 발생 후 재시도 또는 다른 방식으로 전환 시 대기 시간 최소화',
    '인증 수단의 안전성 체감' : '고객이 사용하는 인증 방식의 안전성에 대한 심리적 만족도',
    '이상 거래 탐지(FDS) 알림' : '평소와 다른 환경 접속 시 즉각적인 경고 및 차단 기능의 유효성',
    '비밀번호/패턴 관리 용이성' : '비밀번호 변경 주기 안내 및 복잡성 요구 기준의 합리성',
    '앱 실행 시 보안 점검' : '앱 실행 시 보안 프로그램 작동 상태 및 점검 시간의 적절성',
    '다중 인증(MFA) 옵션' : '고액 거래나 민감 정보 접근 시 추가 인증 옵션의 강제성/선택성',
    '다양한 간편 인증 지원 범위' : '지문, 얼굴(Face ID), 간편 비밀번호, 패턴 등 시장 표준 방식 지원',
    '기존 인증 수단의 유지' : '공동/금융 인증서 등 기존 사용자의 익숙한 인증 방식에 대한 지원 지속',
    '기기 변경/재설치 시 유연성' : '기기 교체나 앱 재설치 시 인증 정보를 쉽게 가져올 수 있는 기능',
    '인증 방식 간의 전환 용이성' : '로그인 화면에서 여러 인증 방식을 쉽게 전환 및 선택할 수 있는 UI/UX',
    '비대면 신원 확인의 편리성' : '신규 가입/재등록 시 신분증 촬영, 계좌 인증 등의 간편성',
    '접속 오류 및 튕김 현상' : '트래픽 집중 시 발생하는 앱 강제 종료 또는 서버 접속 실패 빈도',
    'OS/기기별 호환성' : '최신 및 구형 스마트폰 OS, 특정 기기 모델에서의 정상 작동 여부',
    '네트워크 환경 의존성' : 'Wi-Fi 및 모바일 데이터(LTE/5G) 환경 모두에서 일관된 성능 유지',
    '인증 실패의 비반복성' : '인증 정보를 올바르게 입력했음에도 지속적으로 실패하는 오류 방지',
    '시스템 점검 최소화' : '예정된 시스템 점검으로 인해 로그인 불가능한 시간 최소화',
    '로그인 유지 기능' : '앱을 완전히 종료하지 않았을 때 일정 시간 동안 자동 로그인 유지',
    '자동 로그인 설정의 용이성' : '자동 로그인 기능 설정/해제 및 관리 절차의 단순성',
    '인증 정보 입력의 간편성' : '간편 인증(생체/핀번호) 입력 시 최소한의 터치로 가능하게 설계',
    '비밀번호 찾기/재설정 흐름' : '분실 시 비대면으로 쉽고 빠르게 재설정할 수 있는 사용자 흐름',
    '타 서비스 연동 (SSO)' : '다른 KB 금융 앱(KB Pay 등)과의 연동을 통한 인증 절차 간소화',
    '문제 해결 및 대안 제시 능력': "고객의 복잡하거나 어려운 요청에 대해 신속하고 창의적인 해결책을 제시하는 능력",
}

In [56]:
"""
인입 VOC 세팅
"""
pre_product_service_result = [
    "지문인식", "중장년층",
]
pre_performance_quality_result = [
    "미흡", "보안성", "복잡함"
]

voc1 = {
    "id": "1",
    "content": "휴대폰 지문등록을  변경하고 스타뱅킹 이용시  로그인 방식을  지문인식으로  변경할때  쉽게 찾을수 없었음. 인증서 관리에서 찾아들어  가야하는데 검색창에는 로그인 방법이나 지문등의 단어로는 검색할수 없었음",
    "cxe": "인증 방식 간의 전환 용이성"
}
voc2 = {
    "id": "2",
    "content": "중장년층이 편리하게 사용할수 있으면서 보안이 잘 되면 좋을것같다",
    "cxe": "화면 요소의 접근성"
}
voc3 = {
    "id": "3",
    "content": "로그인 화면은\n직관적이고 좋음\n다만 앱이 전반적으로 산만함",
    "cxe": "명확한 가이드 및 레이아웃"
}
voc4 = {
    "id": "4",
    "content": "보안때문에 인증이 강화좋은데 복잡한 점도 있습니다",
    "cxe": "생체 인증의 인식 속도"
}
voc5 = {
    "id": "5",
    "content": "지문인증이 강화좋은데 복잡한 점도 있습니다2",
    "cxe": "생체 인증의 인식 속도"
}

vocs = [voc1, voc2, voc3, voc4, voc5]

In [51]:
"""
인입 VOC 세팅 (유사 VOC)
"""
pre_product_service_result = [
    "조언", "대안제시",
]
pre_performance_quality_result = [
    "미흡"
]


voc1 = {
    "id": "1",
    "content": "미흡한 조언",
    "cxe": "문제 해결 및 대안 제시 능력"
}
voc2 = {
    "id": "2",
    "content": "조언이 부족함",
    "cxe": "문제 해결 및 대안 제시 능력"
}
voc3 = {
    "id": "3",
    "content": "안되는 이유만제시하고 다른해결방안이나 대안제시가 미흡",
    "cxe": "문제 해결 및 대안 제시 능력"
}

vocs = [voc1, voc2, voc3]

### 3. 개체어 추출

In [4]:
"""
GPT-OSS-120b
세팅
"""

from langchain_openai import ChatOpenAI  
 
llm = ChatOpenAI(  
    model="gpt-oss-120b",  
    base_url="http://stg-llmops-trnn-genaihub.kbonecloud.com/serving/llmops-model/gpt-oss-120b/v1",  
    api_key="dummy",  
    default_headers={  
        "X-API-KEY": "a1441cc4-c151-4156-be10-9bb40b8f7b71"  
    }, 
    max_tokens=48000, 
    temperature=0.2, 
    top_p = 0.9 
)

In [42]:
"""
V1
+ 고객경험요소만
+ 개체어 식별
"""

from langchain.schema import BaseMessage, HumanMessage, SystemMessage

index = 0

for voc in vocs:

    print(f"Summary Generate Index: {index} VOC:")
    print(f"{voc}")
    index = index + 1

    cxe_h = ""
    
    human_prompt = f"""
# 인입 VOC
{voc['content']}

# 고객경험요소:
| 고객경험요소 | 설명 |
|--------------|------|
| {voc['cxe']} | {cxe_dict[voc['cxe']]} |
    """.strip()

    messages = [
        SystemMessage(content=system_prompt_v1_without_attr),
        HumanMessage(content=human_prompt)
    ]

    for chunk in llm.stream(messages):
        print(chunk.content, end="", flush=True)
    print("\n==================================================")

Summary Generate Index: 0 VOC:
{'id': '1', 'content': '미흡한 조언', 'cxe': '문제 해결 및 대안 제시 능력'}
{
    "result": {
        "product_service": "조언",
        "performance_quality": "미흡한"
    }
}
Summary Generate Index: 1 VOC:
{'id': '2', 'content': '조언이 부족함', 'cxe': '문제 해결 및 대안 제시 능력'}
```json
{
    "result": {
        "product_service": "조언",
        "performance_quality": "부족함"
    }
}
```
Summary Generate Index: 2 VOC:
{'id': '3', 'content': '안되는 이유만제시하고 다른해결방안이나 대안제시가 미흡', 'cxe': '문제 해결 및 대안 제시 능력'}
{
    "result": {
        "product_service": "해결방안",
        "performance_quality": "미흡"
    }
}


In [49]:
"""
V2
+ 고객경험요소만
+ 복수 VOC
"""

from langchain.schema import BaseMessage, HumanMessage, SystemMessage

index = 0

voc_ascii_header = (
    "| ID | VOC 원문 | VOC 고객경험요소 |\n"
    "|----|----------|---------------------|"
)

cx_ascii_header = (
    "| 고객경험요소 | 설명 |\n"
    "|--------------|------|"
)

voc_ascii_rows = []
cx_ascii_rows = set()
for voc in vocs:
    print(voc)
    voc_row = f"| {voc['id']} | {repr(voc['content'])} | {voc['cxe']} |"
    voc_ascii_rows.append(voc_row)
    
    cx_row = f"| {voc['cxe']} | {cxe_dict[voc['cxe']]} |"
    cx_ascii_rows.add(cx_row)


voc_ascii_table = "\n".join([voc_ascii_header] + voc_ascii_rows)
cx_ascii_table = "\n".join([cx_ascii_header] + list(cx_ascii_rows))

human_prompt = f"""
# 고객경험요소 사전
{cx_ascii_table}

---

# 인입 VOC 리스트
{voc_ascii_table}
"""



messages = [
    SystemMessage(content=system_prompt_v2_without_attr),
    HumanMessage(content=human_prompt)
]
for chunk in llm.stream(messages):
    print(chunk.content, end="", flush=True)

{'id': '1', 'content': '미흡한 조언', 'cxe': '문제 해결 및 대안 제시 능력'}
{'id': '2', 'content': '조언이 부족함', 'cxe': '문제 해결 및 대안 제시 능력'}
{'id': '3', 'content': '안되는 이유만제시하고 다른해결방안이나 대안제시가 미흡', 'cxe': '문제 해결 및 대안 제시 능력'}
{
  "result": {
    "1": {
      "product_service": "조언",
      "performance_quality": "미흡한"
    },
    "2": {
      "product_service": "조언",
      "performance_quality": "미흡한"
    },
    "3": {
      "product_service": "대안제시",
      "performance_quality": "미흡한"
    }
  }
}

In [57]:
"""
V3
+ 고객경험요소만
+ 복수 VOC
+ 이전 결과 첨부
"""

from langchain.schema import BaseMessage, HumanMessage, SystemMessage

index = 0

voc_ascii_header = (
    "| ID | VOC 원문 | VOC 고객경험요소 |\n"
    "|----|----------|---------------------|"
)

cx_ascii_header = (
    "| 고객경험요소 | 설명 |\n"
    "|--------------|------|"
)

voc_ascii_rows = []
cx_ascii_rows = set()
for voc in vocs:
    print(voc)
    voc_row = f"| {voc['id']} | {repr(voc['content'])} | {voc['cxe']} |"
    voc_ascii_rows.append(voc_row)
    
    cx_row = f"| {voc['cxe']} | {cxe_dict[voc['cxe']]} |"
    cx_ascii_rows.add(cx_row)


voc_ascii_table = "\n".join([voc_ascii_header] + voc_ascii_rows)
cx_ascii_table = "\n".join([cx_ascii_header] + list(cx_ascii_rows))

human_prompt = f"""
# 이전 수행 결과
## 이전 실행 상품·서비스 용어 목록
{pre_product_service_result}

## 이전 실행 성능·품질 용어 목록
{pre_performance_quality_result}
---

# 고객경험요소 사전
{cx_ascii_table}

---

# 인입 VOC 리스트
{voc_ascii_table}
"""



messages = [
    SystemMessage(content=system_prompt_v3),
    HumanMessage(content=human_prompt)
]
for chunk in llm.stream(messages):
    print(chunk.content, end="", flush=True)

{'id': '1', 'content': '휴대폰 지문등록을  변경하고 스타뱅킹 이용시  로그인 방식을  지문인식으로  변경할때  쉽게 찾을수 없었음. 인증서 관리에서 찾아들어  가야하는데 검색창에는 로그인 방법이나 지문등의 단어로는 검색할수 없었음', 'cxe': '인증 방식 간의 전환 용이성'}
{'id': '2', 'content': '중장년층이 편리하게 사용할수 있으면서 보안이 잘 되면 좋을것같다', 'cxe': '화면 요소의 접근성'}
{'id': '3', 'content': '로그인 화면은\n직관적이고 좋음\n다만 앱이 전반적으로 산만함', 'cxe': '명확한 가이드 및 레이아웃'}
{'id': '4', 'content': '보안때문에 인증이 강화좋은데 복잡한 점도 있습니다', 'cxe': '생체 인증의 인식 속도'}
{'id': '5', 'content': '지문인증이 강화좋은데 복잡한 점도 있습니다2', 'cxe': '생체 인증의 인식 속도'}
{
  "result": {
    "1": {
      "product_service": "지문인식",
      "performance_quality": "복잡함"
    },
    "2": {
      "product_service": "중장년층",
      "performance_quality": "보안성"
    },
    "3": {
      "product_service": "로그인 화면",
      "performance_quality": "산만함"
    },
    "4": {
      "product_service": "생체 인증",
      "performance_quality": "복잡함"
    },
    "5": {
      "product_service": "지문인식",
      "performance_quality": "복잡함"
    }
  }
}